# Colab Worker — Corpo e Mente
Qwen3-VL-Embedding-2B
Runtime recomendado: T4 GPU

In [1]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio \
    sentence-transformers langchain-text-splitters \
    opencv-python-headless pillow numpy \
    ultralytics

In [2]:
import ultralytics
print("✅ ultralytics versión:", ultralytics.__version__)

# Verificar que las dependencias previas no se rompieron
import fastapi, uvicorn, cv2, numpy, PIL
from sentence_transformers import SentenceTransformer
print("✅ Dependencias previas intactas")

✅ ultralytics versión: 8.4.166
✅ Dependencias previas intactas


In [3]:
import os
import cv2
import time
import subprocess
import sys
import numpy as np
import requests
from PIL import Image
from pyngrok import ngrok
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse

# Token de ngrok configurado
NGROK_TOKEN = "3JBqQghLHManErBLA80aFdKUBYR_5aTVx17Kzhf5gYVCju9mW"
try:
    from google.colab import userdata
    secret_token = userdata.get("NGROK_TOKEN")
    if secret_token:
        NGROK_TOKEN = secret_token
except Exception:
    pass

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ Token ngrok configurado:", NGROK_TOKEN[:8] + "...")


✅ Token ngrok configurado: 3JBqQghL...


In [4]:
import torch
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ultralytics import YOLO

# --- 1. Estado inicial de VRAM ---
if torch.cuda.is_available():
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- 2. Cargar Qwen (ya estaba) ---
print("\nCargando Qwen3-VL-Embedding-2B...")
qwen_model = SentenceTransformer('Qwen/Qwen3-VL-Embedding-2B')
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
print("✅ Qwen cargado. Dim:", qwen_model.get_sentence_embedding_dimension())

if torch.cuda.is_available():
    print(f"VRAM usada tras Qwen: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras Qwen: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 3. Cargar YOLO26-pose ---
print("\nCargando YOLO26-pose...")
try:
    yolo_pose = YOLO('yolo26n-pose.pt')
    print("✅ YOLO26-pose cargado")
except Exception as e:
    print(f"❌ Error cargando YOLO26-pose: {e}")
    print("DETENTE Y REPORTA ESTE ERROR. No sustituyas por yolo11n-pose.pt sin autorización.")
    raise

if torch.cuda.is_available():
    print(f"VRAM usada tras YOLO-pose: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras YOLO-pose: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 4. Cargar YOLO26-depth (NUEVO) ---
print("\nCargando YOLO26-depth...")
try:
    yolo_depth = YOLO('yolo26n-depth.pt')
    print("✅ YOLO26-depth cargado")
except Exception as e:
    print(f"❌ Error cargando YOLO26-depth: {e}")
    print("DETENTE Y REPORTA ESTE ERROR. No sustituyas por otro modelo sin autorización.")
    raise

if torch.cuda.is_available():
    print(f"VRAM usada tras YOLO-depth: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras YOLO-depth: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 5. Verificar que Qwen sigue funcionando ---
print("\nVerificando que Qwen sigue operativo...")
test_emb = qwen_model.encode(["prueba de coexistencia"], normalize_embeddings=True)
print(f"✅ Qwen responde. Dim del vector: {len(test_emb[0])}")

print("\n✅ Coexistencia triple verificada: Qwen + YOLO26-pose + YOLO26-depth en la misma T4")

VRAM total: 15.64 GB

Cargando Qwen3-VL-Embedding-2B...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

/tmp/ipykernel_17203/2815555974.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("✅ Qwen cargado. Dim:", qwen_model.get_sentence_embedding_dimension())


✅ Qwen cargado. Dim: 2048
VRAM usada tras Qwen: 4.26 GB
VRAM reservada tras Qwen: 4.26 GB

Cargando YOLO26-pose...
✅ YOLO26-pose cargado
VRAM usada tras YOLO-pose: 4.26 GB
VRAM reservada tras YOLO-pose: 4.26 GB

Cargando YOLO26-depth...
✅ YOLO26-depth cargado
VRAM usada tras YOLO-depth: 4.26 GB
VRAM reservada tras YOLO-depth: 4.26 GB

Verificando que Qwen sigue operativo...
✅ Qwen responde. Dim del vector: 2048

✅ Coexistencia triple verificada: Qwen + YOLO26-pose + YOLO26-depth en la misma T4


In [5]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
import os, cv2

app = FastAPI(title="Corpo e Mente Colab Worker")

@app.get("/health")
async def health():
    return {"status": "ok", "model": "Qwen3-VL-Embedding-2B + YOLO26-pose + YOLO26-depth"}

@app.post("/embed_text")
async def embed_text(texto: str = Form(...)):
    chunks = splitter.split_text(texto)
    embeddings = qwen_model.encode(chunks, normalize_embeddings=True)
    return JSONResponse({
        "chunks": chunks,
        "embeddings": embeddings.tolist(),
        "dimension": 2048,
        "chunk_size": 1500,
        "chunk_overlap": 200
    })

@app.post("/extraer_poses")
async def extraer_poses(video: UploadFile = File(...)):
    temp_path = f"/tmp/{video.filename}"
    with open(temp_path, "wb") as f:
        f.write(await video.read())

    try:
        resultados = yolo_pose.predict(source=temp_path, stream=True, verbose=False)
        esqueletos = []
        frame_idx = 0
        for res in resultados:
            if res.keypoints is not None and len(res.keypoints.data) > 0:
                # YOLO26-pose devuelve 17 keypoints (x, y, conf) = 51 floats
                kpts = res.keypoints.data[0].flatten().tolist()
                # Padding a 133 dims para compatibilidad con EsqueletoBiomecanico
                if len(kpts) < 133:
                    kpts.extend([0.0] * (133 - len(kpts)))
                elif len(kpts) > 133:
                    kpts = kpts[:133]
                esqueletos.append({
                    "frame_idx": frame_idx,
                    "keypoints133": kpts,
                    "angulos_articulares": {}
                })
            frame_idx += 1
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    if not esqueletos:
        return JSONResponse(
            {"error": "No se detectaron poses en el video", "total_frames_procesados": frame_idx},
            status_code=400
        )

    return JSONResponse({
        "total_frames_procesados": frame_idx,
        "total_esqueletos": len(esqueletos),
        "esqueletos": esqueletos
    })


@app.post("/extraer_profundidad")
async def extraer_profundidad(
    video: UploadFile = File(...),
    downsample_factor: int = Form(16)
):
    temp_path = f"/tmp/{video.filename}"
    with open(temp_path, "wb") as f:
        f.write(await video.read())

    try:
        resultados = yolo_depth.predict(source=temp_path, stream=True, verbose=False)
        profundidades = []
        frame_idx = 0
        for res in resultados:
            if hasattr(res, "depth") and res.depth is not None:
                depth_np = res.depth.data.cpu().numpy()
                if downsample_factor > 1:
                    depth_downsampled = depth_np[::downsample_factor, ::downsample_factor]
                else:
                    depth_downsampled = depth_np
                
                profundidades.append({
                    "frame_idx": frame_idx,
                    "depth_shape": list(depth_downsampled.shape),
                    "depth_map": depth_downsampled.tolist(),
                    "min_meters": float(depth_np.min()),
                    "max_meters": float(depth_np.max()),
                    "mean_meters": float(depth_np.mean())
                })
            frame_idx += 1
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    if not profundidades:
        return JSONResponse(
            {"error": "No se pudo extraer profundidad del video", "total_frames_procesados": frame_idx},
            status_code=400
        )

    return JSONResponse({
        "total_frames_procesados": frame_idx,
        "total_frames_con_profundidad": len(profundidades),
        "downsample_factor": downsample_factor,
        "profundidades": profundidades
    })


In [6]:
import threading
# 1. Levantar uvicorn en un hilo, forzando IPv4
def _run_uvicorn():
    import asyncio
    from uvicorn.config import Config
    from uvicorn.server import Server

    # Create a uvicorn Config object
    config = Config(app, host="127.0.0.1", port=8000, log_level="warning")
    # Instantiate the server
    server = Server(config=config)
    # Run the server's serve method using the patched asyncio.run,
    # which avoids uvicorn's internal call to asyncio_run with loop_factory.
    asyncio.run(server.serve())

threading.Thread(target=_run_uvicorn, daemon=True).start()

# 2. Esperar a que uvicorn escuche localmente
print("⏳ Esperando a que uvicorn levante...")
uvicorn_ok = False
for i in range(30):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print(f"✅ uvicorn listo en {i+1}s: {r.json()}")
            uvicorn_ok = True
            break
    except Exception:
        pass
    time.sleep(1)

if not uvicorn_ok:
    raise RuntimeError("❌ uvicorn no respondió en 30s. Revisa errores de importación.")

# 3. Conectar ngrok apuntando a IPv4 explícita
ngrok.kill()  # limpiar túneles previos
ngrok_tunnel = ngrok.connect("http://127.0.0.1:8000", bind_tls=True)

print("=" * 60)
print(f"COLAB_TUNNEL_URL = {ngrok_tunnel.public_url}")
print("Copia esta URL en tu backend/.env como COLAB_TUNNEL_URL")
print("=" * 60)

# 4. Probar el túnel público
time.sleep(2)
try:
    r = requests.get(f"{ngrok_tunnel.public_url}/health", timeout=15)
    print(f"🌐 Túnel público → {r.status_code}: {r.text}")
except Exception as e:
    print(f"⚠️ No se pudo probar el túnel público: {e}")
    print("Pero uvicorn está corriendo localmente. Prueba desde tu navegador.")

⏳ Esperando a que uvicorn levante...
✅ uvicorn listo en 2s: {'status': 'ok', 'model': 'Qwen3-VL-Embedding-2B + YOLO26-pose + YOLO26-depth'}
COLAB_TUNNEL_URL = https://oasis-displace-size.ngrok-free.dev
Copia esta URL en tu backend/.env como COLAB_TUNNEL_URL
🌐 Túnel público → 200: {"status":"ok","model":"Qwen3-VL-Embedding-2B + YOLO26-pose + YOLO26-depth"}
